# Formas Fechadas para o Problema GX11 — tempo longo vs tempo curto

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/gx11_formas_fechadas.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Objetivo

Apresentar e comparar **duas formas fechadas equivalentes** para a função de Green do problema X11 (placa finita com temperatura prescrita nas duas faces), evidenciando que uma é eficiente no regime de **tempos longos** e a outra no regime de **tempos curtos**.

Ambas são exatas: derivam uma da outra pela **transformação de soma de Poisson** (identidade de Jacobi para a função theta). Trocá-las conforme o regime de $\tilde{\tau} = \alpha(t-\tau)/L^2$ é o análogo, para o problema X11, do que `x22_formas_fechadas.ipynb` faz com a parte estacionária do X22B10T0.

### Forma 1 — Série de autofunções (tempo longo)

Deduzida no Cap. 4 por separação de variáveis:

$$G^{(\text{L})}_{X11}(x,t \mid x',\tau) = \frac{2}{L}\sum_{n=1}^{\infty} e^{-n^2\pi^2 \tilde{\tau}}\,\sin\!\left(\frac{n\pi x}{L}\right)\sin\!\left(\frac{n\pi x'}{L}\right).$$

### Forma 2 — Série de imagens (tempo curto)

Obtida estendendo a fonte $\delta(x-x')$ em uma sequência de imagens com sinais alternados em $\pm x' + 2kL$, para enforce de $G = 0$ em $x=0,L$:

$$G^{(\text{C})}_{X11}(x,t \mid x',\tau) = \frac{1}{\sqrt{4\pi\alpha(t-\tau)}}\sum_{k=-\infty}^{\infty}\Bigl[\,e^{-(x - x' + 2kL)^2/[4\alpha(t-\tau)]} - e^{-(x + x' + 2kL)^2/[4\alpha(t-\tau)]}\,\Bigr].$$

O objetivo do *notebook* é mostrar **numericamente** que ambas convergem para o mesmo valor e que cada uma economiza ordens de grandeza de modos em seu regime natural.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (7.0, 4.2),
    'font.size':       11,
    'axes.grid':       True,
    'grid.alpha':      0.3,
})

## Parâmetros

Usamos $L = 1$ (sem perda de generalidade); o regime fica caracterizado pelo tempo adimensional

$$\tilde{\tau} = \frac{\alpha\,(t-\tau)}{L^2}.$$

In [ ]:
L     = 1.0
alpha = 1.0       # adimensional: τ̃ = (t - τ)

x_ref  = 0.5 * L  # ponto de avaliação
xp_ref = 0.5 * L  # posição da fonte

## Implementação das duas formas fechadas

Cada função aceita o número de termos $N$ a somar — pela simetria $\pm k$ a série de imagens é truncada em $|k| \le K$ usando $2K+1$ termos.

In [ ]:
def G_modal(x, xp, tau_tilde, N, L=L):
    """Forma 1 — soma parcial dos N primeiros modos (tempo longo)."""
    n  = np.arange(1, N+1)
    bn = n * np.pi / L
    return (2.0/L) * np.sum(np.exp(-(bn*L)**2 * tau_tilde)
                            * np.sin(bn*x) * np.sin(bn*xp))

def G_imagens(x, xp, tau_tilde, K, L=L, alpha=alpha):
    """Forma 2 — soma parcial das imagens |k| <= K (tempo curto).

    O tempo dimensional t - τ entra na fórmula via 4·α·(t-τ) = 4·L²·τ̃.
    """
    if tau_tilde <= 0:
        raise ValueError('τ̃ deve ser positivo')
    quatro_at = 4.0 * L**2 * tau_tilde
    pref = 1.0 / np.sqrt(np.pi * quatro_at)
    k = np.arange(-K, K+1)
    pos = np.exp(-(x - xp + 2*k*L)**2 / quatro_at)
    neg = np.exp(-(x + xp + 2*k*L)**2 / quatro_at)
    return pref * np.sum(pos - neg)

## Verificação cruzada: as duas formas coincidem

Usando muitos termos em cada série, $G^{(\text{L})}$ e $G^{(\text{C})}$ produzem o mesmo valor (a menos do ruído de arredondamento) para qualquer $\tilde{\tau} > 0$. Isso é a manifestação numérica da identidade de Jacobi.

In [ ]:
tau_list = [2.0, 1.0, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]
linhas = []
for tt in tau_list:
    G_L = G_modal  (x_ref, xp_ref, tt, N=200)
    G_C = G_imagens(x_ref, xp_ref, tt, K=200)
    linhas.append((tt, G_L, G_C, abs(G_L - G_C)))

df = pd.DataFrame(linhas, columns=['τ̃', 'G modal (N=200)', 'G imagens (K=200)', '|diferença|'])
pd.set_option('display.float_format', '{:.6e}'.format)
print(df.to_string(index=False))

## Convergência: quantos termos cada forma precisa?

Fixando $x = x' = L/2$ e variando $\tilde{\tau}$, comparamos quantos termos cada série exige para atingir uma tolerância $\varepsilon = 10^{-10}$. A referência cruzada é tomada *da outra forma* (com $N$ ou $K$ grande), evitando privilegiar arbitrariamente uma das representações.

In [ ]:
tol = 1e-10

def n_termos_modal(tt, N_max=400):
    G_alvo = G_imagens(x_ref, xp_ref, tt, K=400)
    for N in range(1, N_max+1):
        if abs(G_modal(x_ref, xp_ref, tt, N) - G_alvo) < tol:
            return N
    return N_max

def k_termos_imagens(tt, K_max=400):
    G_alvo = G_modal(x_ref, xp_ref, tt, N=400)
    for K in range(1, K_max+1):
        if abs(G_imagens(x_ref, xp_ref, tt, K) - G_alvo) < tol:
            return K
    return K_max

tts   = np.array([2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001])
Nmod  = np.array([n_termos_modal   (tt) for tt in tts])
Kimg  = np.array([k_termos_imagens(tt) for tt in tts])

df2 = pd.DataFrame({'τ̃': tts,
                    'N (modal) p/ tol=1e-10': Nmod,
                    'K (imagens) p/ tol=1e-10': Kimg})
pd.set_option('display.float_format', '{:.3g}'.format)
print(df2.to_string(index=False))

In [ ]:
fig, ax = plt.subplots()
ax.loglog(tts, Nmod, '-o',  lw=1.8, label='Forma modal — $N$')
ax.loglog(tts, Kimg, '--s', lw=1.8, label='Forma imagens — $K$')
ax.set_xlabel(r'$\tilde{\tau} = \alpha (t-\tau)/L^2$')
ax.set_ylabel(r'termos para erro $< 10^{-10}$')
ax.set_title('Forma econômica em cada regime — GX11 ($x=x\'=L/2$)')
ax.legend(loc='best', fontsize=9, framealpha=0.9)
ax.invert_xaxis()
plt.tight_layout()
plt.show()

### Leitura do gráfico

- **Tempos longos** ($\tilde{\tau} \gtrsim 0{,}1$): a forma modal converge em pouquíssimos termos (o decaimento $e^{-n^2\pi^2\tilde{\tau}}$ é violento); a forma de imagens demanda muitos.
- **Tempos curtos** ($\tilde{\tau} \lesssim 0{,}05$): inverte-se — a forma de imagens converge em 1–3 termos (a gaussiana é bem localizada), enquanto a modal exigiria centenas.
- O cruzamento ocorre próximo de $\tilde{\tau} \approx 1/\pi^2 \approx 0{,}1$, que é o valor natural sugerido pela identidade de Jacobi para escolher entre as duas formas.

A recomendação prática reproduz a Tabela 4.1 do Beck: programar **ambas** as formas e selecionar em tempo de execução a que for mais econômica para o $\tilde{\tau}$ corrente.

## Perfil espacial em diferentes tempos

Gráfico de $G_{X11}(x, t\mid x'=L/2, 0)$ ao longo da placa, para vários $\tilde{\tau}$, usando em cada caso a forma mais conveniente.

In [ ]:
x = np.linspace(1e-6, L - 1e-6, 401)
tts_plot = [0.001, 0.01, 0.05, 0.1, 0.5]
estilos  = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]

fig, ax = plt.subplots()
for k, tt in enumerate(tts_plot):
    if tt < 0.1:
        G_vals = np.array([G_imagens(xi, 0.5*L, tt, K=8) for xi in x])
        forma  = 'imagens'
    else:
        G_vals = np.array([G_modal  (xi, 0.5*L, tt, N=40) for xi in x])
        forma  = 'modal'
    ax.plot(x/L, G_vals, estilos[k], lw=1.8,
            label=fr'$\tilde{{\tau}}={tt}$ ({forma})')
ax.set_xlabel(r'$x/L$')
ax.set_ylabel(r'$G_{X11}(x,t\mid L/2,0)$')
ax.set_title(r'Perfil de $G_{X11}$ — fonte em $x\'=L/2$')
ax.legend(loc='best', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

## Síntese

1. O problema X11 admite **duas formas fechadas exatas** para a função de Green — modal (tempo longo) e imagens (tempo curto) — ligadas pela identidade de Jacobi.
2. **Nenhuma das duas é "aproximação"** da outra: ambas são exatas no limite $N, K \to \infty$. A escolha é apenas de **economia computacional** em função do regime de $\tilde{\tau}$.
3. O cruzamento ocorre próximo de $\tilde{\tau} \approx 0{,}1$. Implementações de produção devem comutar entre elas nesse valor.
4. Esse mesmo padrão (séries duais ligadas por Poisson/Jacobi) aparece em GX22, GX21, e — com mais cuidado — também em GX13/GX23. Os *notebooks* `x21_formas_fechadas.ipynb` e `x22_formas_fechadas.ipynb` exploram variantes do mesmo princípio.